# Validation: DuckDB access layer vs. the legacy Parquet system

Three refactored things are compared, each for correctness and then for cost:

1. **Filtering** - `provider.df(...)` vs `pd.read_parquet(...)` + boolean mask
2. **Heatmap** - `prepare_equal_bins_heatmap_sql` vs per-slot `np.digitize` binning
3. **Plots** - `create_axiswise_plots2` and `plot_digital_twin_heatmap_gradient`, fed from each side

Compared against `DBold.duckdb`, the DuckDB build of the same Parquet files.
`DBnew.duckdb` is a later campaign (plate 40, no `Oscilloscope` origin) that the legacy
system never processed, so it has no baseline to compare against.

Timings: each measurement runs in a fresh child process (`ru_maxrss` is a high-water mark),
first run discarded, median of the rest. Warm OS page cache.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in Path.cwd().resolve().parents if (p / "validation_data_access").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from validation_data_access.tests import harness

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)

if not harness.data_status()["ready"]:
    raise SystemExit("Parquet files or DBold.duckdb not present; see VALIDATION.md.")

s = harness.load_stacks()
provider, new_dp, new_viz, legacy_viz = s["provider"], s["new_dp"], s["new_viz"], s["legacy_viz"]

verdict, timings = [], []

## 1. Filtering

`provider.df(plate, slot, ...)` vs reading the slot's Parquet file and masking.

### 1a. Correctness

Compared as multisets of rows, not row-by-row: a SQL result set has no inherent order, and
sorting by a subset of columns leaves ties the two stacks break differently. Numeric
columns are canonicalised first, because DuckDB returns some as nullable integers where
Parquet returns floats, and because Parquet preserves the sign bit of a zero.

In [ ]:
rows = []
for plate, slot in harness.SLOT_CASES:
    legacy_df = harness.legacy_slot_df(plate, slot)
    new_df = provider.df(plate, slot)
    columns = sorted(legacy_df.columns)

    result = harness.frame_equality(legacy_df, new_df, columns=columns)
    rows.append(
        {
            "plate": plate,
            "slot": slot,
            "rows_legacy": result["rows_legacy"],
            "rows_new": result["rows_new"],
            "rows_only_legacy": result["rows_only_legacy"],
            "rows_only_new": result["rows_only_new"],
            "columns": len(columns),
            "identical": result["ok"],
        }
    )

full_frames = pd.DataFrame(rows)
verdict.append(
    {
        "check": "filtering: whole slot, all columns",
        "passed": bool(full_frames["identical"].all()),
        "detail": f"{full_frames['rows_legacy'].sum():,} rows over {len(full_frames)} slots",
    }
)
full_frames

In [ ]:
# The two columns whose storage type differs. Same values, better typed in the database.
legacy_df = harness.legacy_slot_df(*harness.SLOT_CASES[0])
new_df = provider.df(*harness.SLOT_CASES[0])
dtypes = harness.dtype_table(legacy_df, new_df)
dtypes[dtypes["legacy_dtype"] != dtypes["new_dtype"]]

In [ ]:
# The narrow query both stacks are timed on below returns the same rows.
legacy_hit, _ = harness.legacy_targeted_query(**harness.TARGETED_QUERY)
new_hit = provider.df(
    harness.TARGETED_QUERY["plate"],
    harness.TARGETED_QUERY["slot"],
    data_origin=harness.TARGETED_QUERY["data_origin"],
    signals=harness.TARGETED_QUERY["signals"],
    wcs_min=harness.TARGETED_QUERY["wcs_min"],
    wcs_max=harness.TARGETED_QUERY["wcs_max"],
)
narrow = harness.frame_equality(legacy_hit, new_hit, columns=sorted(legacy_hit.columns))
verdict.append(
    {
        "check": "filtering: narrow query",
        "passed": narrow["ok"],
        "detail": f"{narrow['rows_legacy']} rows returned by both",
    }
)
pd.Series({k: v for k, v in narrow.items() if k != "columns"})

### 1b. Cost

`rows_read` is how many rows had to be materialised in Python to produce the answer. On
the legacy side that is the whole file regardless of how narrow the question is.

In [ ]:
filtering_timings = [
    harness.measure_pair(
        "legacy_targeted_query", "new_targeted_query", harness.TARGETED_QUERY,
        label="narrow query, HF_Data by Signal", repeats=5,
    ),
    harness.measure_pair(
        "legacy_targeted_query_axis", "new_targeted_query_axis", harness.TARGETED_QUERY_AXIS,
        label="narrow query, Oscilloscope by Axis", repeats=5,
    ),
]
timings += filtering_timings
harness.results_frame(filtering_timings)

## 2. Heatmap

`prepare_equal_bins_heatmap_sql` (one SQL statement) vs the legacy path: open every
Parquet file of the plate, bin each slot with `np.digitize` over
`np.arange(0, y_max + bin, bin)`, concatenate, normalise.

Also compared: the two values that anchor the ends of the colour gradient
(`get_min_max_amplitudes_sql_from_db` vs the `idxmin`/`idxmax` lookup in the legacy
widget). An identical bin table still renders differently if those move.

In [ ]:
rows, heatmaps = [], {}
for plate in harness.PLATES:
    legacy_hm = harness.legacy_heatmap(plate, harness.BIN_SIZE_MM)
    new_hm = new_dp.prepare_equal_bins_heatmap_sql(
        plate,
        bin_size_mm=harness.BIN_SIZE_MM,
        target_signal=harness.TARGET_SIGNAL,
        target_origin=harness.TARGET_ORIGIN,
        compute_normalized_global=True,
    )
    heatmaps[plate] = (legacy_hm, new_hm)

    report = harness.compare_frames(
        legacy_hm, new_hm,
        join_keys=["Nut", "Y_min"],
        numeric_cols=["Y_max", "Y_bin_center", "RMS_raw", "RMS_normalized_global"],
        atol=1e-9, rtol=1e-9,
    )

    legacy_min, legacy_max = harness.legacy_amplitude_anchors(legacy_hm, plate)
    new_min, new_max = new_dp.get_min_max_amplitudes_sql_from_db(
        plate, bin_size_mm=harness.BIN_SIZE_MM,
        target_signal=harness.TARGET_SIGNAL, target_origin=harness.TARGET_ORIGIN,
    )

    rows.append(
        {
            "plate": plate,
            **{k: v for k, v in harness.report_row("", report).items() if k not in ("check", "detail")},
            "anchors_identical": abs(legacy_min - new_min) < 1e-12 and abs(legacy_max - new_max) < 1e-12,
        }
    )

heatmap_parity = pd.DataFrame(rows)
verdict.append(
    {
        "check": "heatmap: bins and colour anchors",
        "passed": bool(heatmap_parity["passed"].all() and heatmap_parity["anchors_identical"].all()),
        "detail": f"{heatmap_parity['rows_legacy'].sum()} bins over {len(harness.PLATES)} plates, "
        f"max |diff| {heatmap_parity['max_abs_diff'].max():.1e}",
    }
)
heatmap_parity

The two implementations reach the same result through unrelated mechanisms in four places
where they could have diverged. The counts below show those cases are actually exercised
rather than simply absent - except `Value IS NULL`, the one case where the two genuinely
disagree, which never occurs in this data.

In [ ]:
table = provider.table()
rows = []
for plate in harness.PLATES:
    counts = provider.query_df(
        f"""
        SELECT count(*) AS total,
               count(*) FILTER (WHERE WCS_Y_mm < 0)  AS negative_y,
               count(*) FILTER (WHERE Value IS NULL) AS null_value
        FROM {table} WHERE Platte = ? AND Axis = ? AND DataOrigin = ?
        """,
        [plate, harness.TARGET_SIGNAL, harness.TARGET_ORIGIN],
    ).iloc[0]

    at_max = provider.query_df(
        f"""
        WITH d AS (
            SELECT Nut, WCS_Y_mm FROM {table}
            WHERE Platte = ? AND Axis = ? AND DataOrigin = ?
              AND Nut IS NOT NULL AND WCS_Y_mm IS NOT NULL AND Value IS NOT NULL AND WCS_Y_mm >= 0
        ),
        m AS (SELECT Nut, max(WCS_Y_mm) AS y_max FROM d GROUP BY Nut)
        SELECT count(*) FROM d JOIN m USING (Nut) WHERE d.WCS_Y_mm = m.y_max
        """,
        [plate, harness.TARGET_SIGNAL, harness.TARGET_ORIGIN],
    ).iloc[0, 0]

    rows.append(
        {
            "plate": plate,
            "rows_in_scope": int(counts["total"]),
            "WCS_Y_mm < 0": int(counts["negative_y"]),
            "at slot y_max": int(at_max),
            "Value IS NULL": int(counts["null_value"]),
        }
    )
pd.DataFrame(rows)

In [ ]:
heatmap_timings = [
    harness.measure_pair(
        "legacy_heatmap", "new_heatmap",
        {
            "plate": plate,
            "bin_size_mm": harness.BIN_SIZE_MM,
            "target_signal": harness.TARGET_SIGNAL,
            "target_origin": harness.TARGET_ORIGIN,
        },
        label=f"heatmap, plate {plate} ({len(harness.plate_files(plate))} slots)", repeats=5,
    )
    for plate in harness.PERF_PLATES
]
timings += heatmap_timings
harness.results_frame(heatmap_timings)

## 3. Plots

Figures are compared by signature, not by pixels: per trace, the count, minimum, maximum
and mean of every coordinate array, plus shape and annotation counts.

### 3a. Axis-wise plots

`create_axiswise_plots2` returns one figure per machine axis. The legacy version filters
by `DataOrigin` itself; the new one expects a frame already filtered by the query.

It also downsamples to `max_display_points` before plotting, and `downsample_df` selects
**positionally** (`np.linspace`), so the sampled points depend on row order - which
neither system defines. Run once with downsampling off (does the *data* agree?) and once
at the shipped default (does the *figure* agree?).

In [ ]:
NO_DOWNSAMPLING = 10**9
rows = []
for plate, slot in harness.SLOT_CASES:
    legacy_raw = harness.legacy_slot_df(plate, slot)
    new_prefiltered = provider.axiswise_plot_df(plate, slot, data_origin=harness.PLOT_ORIGIN)

    for label, max_points in [("all points", NO_DOWNSAMPLING), ("downsampled to 10,000", 10_000)]:
        legacy_figs = legacy_viz.create_axiswise_plots2(
            legacy_raw, filter_value=harness.PLOT_ORIGIN, normalize_method=None, max_display_points=max_points
        )
        new_figs = new_viz.create_axiswise_plots2(
            new_prefiltered, normalize_method=None, max_display_points=max_points
        )

        shared = sorted(set(legacy_figs) & set(new_figs))
        identical, sampling_only, stats = 0, 0, set()
        for axis in shared:
            diff = harness.compare_signatures(
                harness.figure_signature(legacy_figs[axis]), harness.figure_signature(new_figs[axis])
            )
            identical += diff["ok"]
            sampling_only += diff["sampling_only"]
            stats |= diff["stats_differing"]

        rows.append(
            {
                "plate": plate, "slot": slot, "variant": label, "axes": len(shared),
                "identical": identical,
                "differing_by_sampling_only": sampling_only,
                "statistics_differing": ", ".join(sorted(stats)) or "-",
            }
        )

axiswise = pd.DataFrame(rows)
axiswise

With every point plotted, all axes match. After downsampling some do not - and the
difference is confined to `y.mean`: every trace keeps its point count, minimum and
maximum. That is two different samples of one dataset, not two datasets.

The reason `Time` is not enough to pin down row order:

In [ ]:
frame = provider.axiswise_plot_df("26", 20, data_origin=harness.PLOT_ORIGIN)
ties = (
    frame.groupby(["Axis", "Signal"])
    .agg(rows=("Time", "size"), distinct_times=("Time", "nunique"))
    .assign(rows_sharing_a_timestamp=lambda d: d["rows"] - d["distinct_times"])
    .sort_values("rows_sharing_a_timestamp", ascending=False)
)
print(f"signals where Time is not unique: {int((ties['rows_sharing_a_timestamp'] > 0).sum())} of {len(ties)}")
ties.head(4)

In [ ]:
all_points = axiswise[axiswise["variant"] == "all points"]
downsampled = axiswise[axiswise["variant"] == "downsampled to 10,000"]

verdict.append(
    {
        "check": "plots: axis-wise",
        "passed": bool(
            (all_points["identical"] == all_points["axes"]).all()
            and (downsampled["identical"] + downsampled["differing_by_sampling_only"] == downsampled["axes"]).all()
        ),
        "detail": f"all {int(all_points['axes'].sum())} axes identical with every point plotted; "
        f"{int(downsampled['differing_by_sampling_only'].sum())} differ in y.mean only after downsampling",
    }
)
verdict[-1]

### 3b. Rendered heatmap

The new visualizer coalesces the Qw iso-lines from one `line` shape per segment into one
SVG `path` per contiguous run, so shape *counts* differ by design. The comparison decodes
the path strings back into segments and compares the geometry.

In [ ]:
rows = []
for plate in harness.PLATES:
    legacy_hm, new_hm = heatmaps[plate]
    legacy_fig = legacy_viz.plot_digital_twin_heatmap_gradient(legacy_hm, df_summary=harness.legacy_summary(plate))
    new_fig = new_viz.plot_digital_twin_heatmap_gradient(new_hm, df_summary=new_dp.summarize_chatter_cases_sql(plate))

    geometry = harness.compare_polylines(legacy_fig, new_fig)
    signature = harness.compare_signatures(harness.figure_signature(legacy_fig), harness.figure_signature(new_fig))
    other_legacy = geometry["shapes_legacy"] - geometry["segments_legacy"]
    other_new = sum(1 for sh in new_fig.layout.shapes if sh.line.dash != "dot")

    rows.append(
        {
            "plate": plate,
            "traces": f"{len(legacy_fig.data)} / {len(new_fig.data)}",
            "trace_data_max_diff": signature["max_abs_diff"],
            "non_isoline_shapes": f"{other_legacy} / {other_new}",
            "isoline_shapes": f"{geometry['shapes_legacy'] - other_legacy} / {geometry['shapes_new'] - other_new}",
            "isoline_segments": f"{geometry['segments_legacy']} / {geometry['segments_new']}",
            "geometry_identical": geometry["ok"],
        }
    )

heatmap_figures = pd.DataFrame(rows)
verdict.append(
    {
        "check": "plots: rendered heatmap",
        "passed": bool(heatmap_figures["geometry_identical"].all() and (heatmap_figures["trace_data_max_diff"] == 0).all()),
        "detail": "trace data identical; iso-lines re-encoded as SVG paths, same geometry",
    }
)
heatmap_figures

In [ ]:
plate = harness.FIGURE_PLATE
legacy_hm, new_hm = heatmaps[plate]
legacy_fig = legacy_viz.plot_digital_twin_heatmap_gradient(legacy_hm, df_summary=harness.legacy_summary(plate))
new_fig = new_viz.plot_digital_twin_heatmap_gradient(new_hm, df_summary=new_dp.summarize_chatter_cases_sql(plate))

written = harness.save_comparison_png(
    legacy_fig, new_fig,
    harness.RESULTS_DIR / "figures" / f"heatmap_plate_{plate}.png",
    title=f"Digital twin heatmap, plate {plate}, bin size {harness.BIN_SIZE_MM:g} mm",
)
print(written["png"] if written["written"] else f"PNG export failed: {written['error']}")
new_fig

`axiswise_plot_df` gained an optional `max_points_per_signal` parameter that reduces each
`(Nut, Signal)` group inside the query via `ROW_NUMBER()`, instead of transferring every row
and downsampling in Python afterwards (`viz/visualizer.py::downsample_df`). Both the existing
full-fetch path and the new pushed-down path are measured below; the production plotting path
in `viz/visualizer.py` is unchanged in this revision.

In [ ]:
plot_timings = [
    harness.measure_pair(
        "legacy_plot_df", "new_plot_df",
        {"plate": plate, "slot": slot, "data_origin": harness.PLOT_ORIGIN},
        label=f"plot data (full fetch), slot {plate}/{slot}", repeats=5,
    )
    for plate, slot in harness.PERF_SLOTS
] + [
    harness.measure_pair(
        "legacy_plot_df", "new_plot_df_pushed_down",
        {
            "plate": plate, "slot": slot, "data_origin": harness.PLOT_ORIGIN,
            "max_points_per_signal": harness.MAX_DISPLAY_POINTS,
        },
        label=f"plot data (SQL-side downsample to {harness.MAX_DISPLAY_POINTS:,}), slot {plate}/{slot}",
        repeats=5,
    )
    for plate, slot in harness.PERF_SLOTS
]
timings += plot_timings
harness.results_frame(plot_timings)

## Summary

In [ ]:
verdict_df = pd.DataFrame(verdict)
timings_df = harness.results_frame(timings)

harness.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
verdict_df.to_csv(harness.RESULTS_DIR / "correctness.csv", index=False)
timings_df.to_csv(harness.RESULTS_DIR / "performance.csv", index=False)
heatmap_parity.to_csv(harness.RESULTS_DIR / "correctness_heatmap.csv", index=False)
full_frames.to_csv(harness.RESULTS_DIR / "correctness_filtering.csv", index=False)
axiswise.to_csv(harness.RESULTS_DIR / "correctness_axiswise.csv", index=False)

print(f"all checks passed: {verdict_df['passed'].all()}")
verdict_df

In [ ]:
timings_df[
    ["task", "legacy_seconds", "new_seconds", "speedup",
     "legacy_peak_rss_mb", "new_peak_rss_mb", "legacy_rows_read", "new_rows_returned"]
].round(3)

Fetching a slot for plotting is the one case where the new stack is slower: nothing can be
pushed down when the answer is the whole slot, and DuckDB reconstructs rows from columnar
storage that Parquet hands over almost directly.